# 🌳 คำอธิบายและตัวอย่างการปฏิบัติการต้นไม้ตัดสินใจ (Decision Trees)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **ต้นไม้ตัดสินใจ (Decision Trees)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลองแบบ 3 คลาส อ้างอิงตามกรณีศึกษากล่อง Bounding Box ได้แก่ `valve` (วาล์ว), `flange` (แปลนท่อ) และ `small-fitting` (ข้อต่อขนาดเล็ก)
2. ฝึกสอนแบบจำลอง **Decision Tree Classifier** โดยใช้ไลบรารี `scikit-learn`
3. จำลองภาพขอบเขตการตัดสินใจแบบแนวตั้งฉากตามแกนพิกัด (Axis-aligned Orthogonal Decision Boundaries)
4. แสดงโครงสร้างภาพผังแผนภูมิต้นไม้ตัดสินใจ
5. พัฒนาแบบจำลอง **Decision Tree จากศูนย์ (from scratch)** ด้วยภาษา Python/NumPy โดยอ้างอิงตามทฤษฎี **เอนโทรปี (Entropy)** และ **การได้ข้อมูล (Information Gain)**
6. ประเมินผลโค้ดที่สร้างขึ้นเองเปรียบเทียบกับไลบรารีมาตรฐาน scikit-learn
7. เชื่อมโยงแนวคิดของต้นไม้ตัดสินใจสู่ขั้นตอนกระบวนการกรองข้อมูลส่วนหลัง (Post-processing Routing Pipelines) ในงานระบบภาพคอมพิวเตอร์ (Computer Vision)

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
from collections import Counter

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การสร้างข้อมูลตามกรณีศึกษา (Case Study Data Generation)

เราจะสุ่มสร้างคุณลักษณะของกล่อง Bounding Box จำนวน 90 ตัวอย่าง ประกอบไปด้วย 2 คุณลักษณะ:
1.  `area` พื้นที่ของกล่องข้อความ (ในหน่วยพิกเซล มีค่าระหว่าง 2,000 ถึง 30,000)
2.  `aspect_ratio` อัตราส่วนความกว้างต่อความสูง (Aspect Ratio มีค่าระหว่าง 0.5 ถึง 2.5)

เกณฑ์ในการแบ่งประเภทคลาส:
*   คลาส 0 (`valve` วาล์ว): พื้นที่ > 15,000 และ อัตราส่วนภาพ > 1.2
*   คลาส 1 (`flange` แปลนท่อ): พื้นที่ > 15,000 และ อัตราส่วนภาพ <= 1.2
*   คลาส 2 (`small-fitting` ข้อต่อท่อขนาดเล็ก): พื้นที่ <= 15,000

In [ ]:
m = 90

# สุ่มสร้างคุณลักษณะแบบสุ่ม
areas = np.random.rand(m, 1) * 28000 + 2000
aspect_ratios = np.random.rand(m, 1) * 2.0 + 0.5
X = np.hstack((areas, aspect_ratios))

# กำหนดป้ายกำกับคลาสตามกฎเกณฑ์ที่ตั้งไว้
y = np.zeros(m, dtype=int)
for i in range(m):
    area, aspect = X[i, 0], X[i, 1]
    if area <= 15000:
        y[i] = 2  # Small fitting
    elif aspect > 1.2:
        y[i] = 0  # Valve
    else:
        y[i] = 1  # Flange

# พล็อตกราฟจุดกระจายตัวของข้อมูล
plt.figure(figsize=(8, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='blue', label='Class 0: Valve', alpha=0.7)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='red', label='Class 1: Flange', alpha=0.7)
plt.scatter(X[y == 2, 0], X[y == 2, 1], color='green', label='Class 2: Small Fitting', alpha=0.7)
plt.axvline(15000, color='black', linestyle='--', alpha=0.5, label='Area Threshold')
plt.axhline(1.2, color='gray', linestyle='--', alpha=0.5, label='Aspect Ratio Threshold')
plt.xlabel('Area (pixels)')
plt.ylabel('Aspect Ratio')
plt.title('PTT Object Classification Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. การสร้าง Decision Tree ด้วย Scikit-Learn

เรามาลองเทรนโมเดลจำแนกประเภทด้วย scikit-learn โดยจำกัดความลึกสูงสุดของต้นไม้ `max_depth=3`

In [ ]:
# ฝึกสอนแบบจำลอง
clf = DecisionTreeClassifier(max_depth=3, criterion='entropy')
clf.fit(X, y)

# ประเมินผลค่าความถูกต้องบนชุดข้อมูลเทรน
y_pred_sklearn = clf.predict(X)
print(f"Training Accuracy: {accuracy_score(y, y_pred_sklearn) * 100:.2f}%")

มาพล็อตภาพไดอะแกรมของสถาปัตยกรรมต้นไม้ตัดสินใจที่เรียนรู้ได้ด้วยฟังก์ชัน `plot_tree` กันครับ

In [ ]:
plt.figure(figsize=(12, 8))
plot_tree(clf, feature_names=['area', 'aspect_ratio'], class_names=['Valve', 'Flange', 'Small Fitting'], filled=True, rounded=True)
plt.title('Learned Decision Tree Diagram')
plt.show()

## 3. การสร้าง Decision Tree จากศูนย์ (Decision Tree from Scratch)

เราจะพัฒนาแบบจำลองต้นไม้ตัดสินใจด้วยกระบวนการเรียกตัวเองแบบย้อนกลับ (Recursive Node Splitting)
ในแต่ละทางแยกของโหนด เราจะเลือกคุณลักษณะและขีดจำกัดจุดแบ่งที่ให้ค่า **การได้ข้อมูล (Information Gain: IG)** สูงสุด:
$$IG(S, A) = H(S) - \left( \frac{|S_{\text{left}}|}{|S|} H(S_{\text{left}}) + \frac{|S_{\text{right}}|}{|S|} H(S_{\text{right}}) \right)$$

โดยกำหนดให้ $H(S)$ คือค่าความไม่ระเบียบหรือ เอนโทรปี (Entropy) ของชุดข้อมูล $S$ คำนวณได้ดังนี้:
$$H(S) = -\sum p_i \log_2 p_i$$

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       # ดัชนีคุณลักษณะที่ใช้ทำจุดแยก
        self.threshold = threshold   # ค่าขีดจำกัดจุดแยก
        self.left = left             # โหนดลูกฝั่งซ้าย
        self.right = right           # โหนดลูกฝั่งขวา
        self.value = value           # ป้ายกำกับคลาสคำตอบสุดท้าย (หากเป็นโหนดใบไม้)

    def is_leaf(self):
        return self.value is not None

class CustomDecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.root = None

    def _entropy(self, y):
        counts = np.bincount(y)
        probs = counts / len(y)
        return -np.sum([p * np.log2(p) for p in probs if p > 0])

    def _information_gain(self, y, X_col, threshold):
        parent_entropy = self._entropy(y)
        
        left_idx = np.where(X_col <= threshold)[0]
        right_idx = np.where(X_col > threshold)[0]
        
        if len(left_idx) == 0 or len(right_idx) == 0:
            return 0
            
        n = len(y)
        n_l, n_r = len(left_idx), len(right_idx)
        e_l, e_r = self._entropy(y[left_idx]), self._entropy(y[right_idx])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r
        
        return parent_entropy - child_entropy

    def _best_split(self, X, y):
        best_gain = -1
        split_idx, split_thresh = None, None
        m, n = X.shape
        
        for feature in range(n):
            X_col = X[:, feature]
            thresholds = np.unique(X_col)
            for threshold in thresholds:
                gain = self._information_gain(y, X_col, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feature
                    split_thresh = threshold
                    
        return split_idx, split_thresh

    def _build_tree(self, X, y, depth=0):
        m, n = X.shape
        n_labels = len(np.unique(y))
        
        if depth >= self.max_depth or n_labels == 1 or m < 2:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
            
        split_idx, split_thresh = self._best_split(X, y)
        if split_idx is None:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
            
        left_idx = np.where(X[:, split_idx] <= split_thresh)[0]
        right_idx = np.where(X[:, split_idx] > split_thresh)[0]
        
        left_child = self._build_tree(X[left_idx, :], y[left_idx], depth + 1)
        right_child = self._build_tree(X[right_idx, :], y[right_idx], depth + 1)
        
        return Node(feature=split_idx, threshold=split_thresh, left=left_child, right=right_child)

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_row(self, node, x):
        if node.is_leaf():
            return node.value
            
        if x[node.feature] <= node.threshold:
            return self._predict_row(node.left, x)
        return self._predict_row(node.right, x)

    def predict(self, X):
        return np.array([self._predict_row(self.root, x) for x in X])

# ฝึกสอนแบบจำลองที่เราทำขึ้นมาเอง
tree_scratch = CustomDecisionTree(max_depth=3)
tree_scratch.fit(X, y)

y_pred_scratch = tree_scratch.predict(X)
print(f"Scratch Decision Tree Accuracy: {accuracy_score(y, y_pred_scratch) * 100:.2f}%")

## 4. การแสดงขอบเขตการตัดสินใจ (Decision Boundaries Visualization)

เราลองมาวาดภาพสังเกตพฤติกรรมว่าแบบจำลอง Custom Decision Tree ของเรา มีวิธีกรีดแผ่นดินแบ่งแยกพื้นที่ดินแดนของคุณลักษณะออกเป็นขอบเขตกล่องสี่เหลี่ยมย่อยๆ อย่างไรบ้าง

In [ ]:
from matplotlib.colors import ListedColormap

# สร้างจุดโครงข่ายข้อมูล Grid
x_min, x_max = X[:, 0].min() - 2000, X[:, 0].max() + 2000
y_min, y_max = X[:, 1].min() - 0.2, X[:, 1].max() + 0.2
xx, yy = np.meshgrid(np.arange(x_min, x_max, 100),
                     np.arange(y_min, y_max, 0.01))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# ทำนายผลบนพื้นที่ Grid ด้วยแบบจำลองที่เราทำเอง
Z = tree_scratch.predict(grid_points)
Z = Z.reshape(xx.shape)

# พล็อตกราฟแบ่งขอบเขตพื้นที่การตัดสินใจ
plt.figure(figsize=(10, 6))
cmap_light = ListedColormap(['#AAAAFF', '#FFAAAA', '#AAFFAA'])
cmap_bold = ['blue', 'red', 'green']

plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.5)
for c in [0, 1, 2]:
    plt.scatter(X[y == c, 0], X[y == c, 1], color=cmap_bold[c], label=f'Class {c}', edgecolor='k')

# วาดเส้นแบ่งจุดแยกที่ตรวจพบโดยต้นไม้แบบกำหนดเอง
root_feat = tree_scratch.root.feature
root_thresh = tree_scratch.root.threshold
feat_name = 'Area' if root_feat == 0 else 'Aspect Ratio'
print(f"Root Split: {feat_name} <= {root_thresh:.2f}")

plt.xlabel('Area')
plt.ylabel('Aspect Ratio')
plt.title('Custom Decision Tree Partition Boundaries')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 💡 ความเชื่อมโยงสู่ Deep Learning และ YOLO
*   **การคัดกรองข้อมูลกล่องผลลัพธ์ขั้นตอนหลัง (Bounding Box Post-processing Routing):** ต้นไม้ตัดสินใจมีประโยชน์อย่างยิ่งในการช่วยสร้างกฎเกณฑ์การตัดสินใจแบบฮิวริสติก (Heuristic Rules) ที่รวดเร็ว ตัวอย่างเช่น หากเราต้องติดตั้งตัวตรวจจับ YOLO ให้รันบนชิปสมองกลฝังตัวกำลังต่ำ (เช่น Raspberry Pi หรือ Microcontrollers) การเรียกโครงข่ายประสาทเทียมขนาดใหญ่ซ้ำซ้อนจะสร้างภาระงานหนักและส่งผลให้ล่าช้า แต่หากเราใช้ YOLO เพื่อหาจุดกล่องพิกัดออกมาอย่างเดียว จากนั้นป้อนพิกัดเหล่านั้นผ่านโค้ดสคริปต์ **Decision Tree** ตัวเล็กๆ เพื่อช่วยประเมินการกรองสิ่งรบกวน (เช่น ถ้า `Area <= 15000` ให้กรองออกทันที) หรือวิเคราะห์โครงสร้างเส้นทางต่อยอด ก็จะช่วยให้ระบบทำงานได้เสร็จสิ้นอย่างรวดเร็วโดยไม่ต้องพึ่งพาโครงข่ายประสาทขนาดใหญ่อีกตัว